# Session 4 — Collective Communication
## From Tensor Semantics to Manual Gradient Synchronization

**How to Build Distributed AI Systems**

In the previous session, we learned how NCCL acts as the communication engine: it initializes communicators, discovers topology, selects transports, and organizes work into chunks and channels.

In this session, we change the question from:

> **How can GPUs communicate?**

to:

> **What communication pattern do we actually need?**

We will explore collective operations using **PyTorch Distributed + NCCL + 2 GPUs**.

---

### Lab Environment

This notebook is designed for an environment with:

- Linux
- PyTorch
- CUDA
- **2 NVIDIA GPUs**
- NCCL backend
- `torchrun`

A Kaggle notebook with **2× T4 GPUs** is a good fit.

> **Instructor note:** The goal is not to train a large model. We intentionally use tiny tensors so that the communication semantics are easy to see.


## Learning Outcomes

By the end of this session, you should be able to:

1. Explain the difference between `Broadcast`, `Reduce`, `AllReduce`, `AllGather`, `ReduceScatter`, and `AllToAll`.
2. Predict how tensors will look on each rank **before running the code**.
3. Use `torch.distributed` and `torchrun` to execute collective communication across multiple GPUs.
4. Understand the **Collective Contract** and why mismatched collectives can cause hangs.
5. Explain the role of `dist.barrier()` as a synchronization primitive.
6. Connect `AllReduce` to gradient synchronization in distributed training.
7. Understand the semantic decomposition:

```text
AllReduce = ReduceScatter + AllGather
```

8. Implement **manual gradient synchronization** before moving to DDP.
9. Distinguish between:

```text
WHAT communication result do I want?
```

and:

```text
HOW does NCCL implement that communication?
```


## Session Map

```text
Environment Check
      ↓
Distributed Execution Recap
      ↓
Collective Contract
      ↓
Barrier
      ↓
Broadcast
      ↓
Reduce
      ↓
AllReduce
      ↓
AllGather
      ↓
ReduceScatter
      ↓
AllToAll
      ↓
Ring AllReduce Mental Model
      ↓
Manual Gradient Synchronization
      ↓
Bridge to DDP
```

### Scope

We will focus on the core collectives. The following topics are intentionally kept as short notes for later sessions:

- `Gather` / `Scatter`
- `async_op=True`
- Custom process groups
- Communication benchmarking
- Advanced overlap with CUDA streams


## 1. Environment Check

First, verify that PyTorch can see CUDA and that the runtime exposes two GPUs.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version used by PyTorch:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {props.name} | "
        f"VRAM={props.total_memory / 1024**3:.2f} GB"
    )

In [ ]:
# Optional hardware check
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

> **Expected:** `GPU count` should be 2 for this lab.

If only one GPU is available, you can still study the notebook and understand the semantics, but the NCCL demos below require multiple processes and GPUs.


## 2. Quick Recap — `torchrun`, Rank, Local Rank, GPU

When we run:

```bash
torchrun --standalone --nproc_per_node=2 program.py
```

`torchrun` launches **2 independent Python processes**.

Conceptually:

```text
Process 0
RANK=0
LOCAL_RANK=0
        ↓
      GPU 0


Process 1
RANK=1
LOCAL_RANK=1
        ↓
      GPU 1


WORLD_SIZE=2
```

Each process typically executes:

```python
torch.cuda.set_device(local_rank)
dist.init_process_group(backend="nccl")
```

The two processes then become members of the same distributed process group.

> **Important:** A rank is a software identity inside a distributed group. It is **not** a hardware ID embedded in the GPU.


## 3. Why Are We Creating a Small Runtime Script from the Notebook?

Jupyter normally runs a single Python process, while `torchrun` needs to launch multiple independent Python processes.

To keep the notebook interactive while still executing NCCL correctly, the notebook writes a temporary runtime helper named:

```text
_session4_runtime.py
```

Each section can then run one isolated demo:

```bash
torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo all_reduce
```

This helper is only part of the lab. After the concepts are clear, we can build a cleaner final `collectives_demo.py` separately.


In [ ]:
%%writefile _session4_runtime.py
import os
import time
import argparse

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


def setup():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")

    device = torch.device(f"cuda:{local_rank}")
    return rank, local_rank, world_size, device


def cleanup():
    if dist.is_initialized():
        dist.destroy_process_group()


def fmt(tensor):
    return tensor.detach().cpu().tolist()


def ordered_print(rank, world_size, message):
    """Print rank outputs in deterministic rank order."""
    for r in range(world_size):
        dist.barrier()
        if rank == r:
            print(message, flush=True)
    dist.barrier()


def demo_identity(rank, local_rank, world_size, device):
    ordered_print(
        rank,
        world_size,
        f"Rank {rank} | LOCAL_RANK={local_rank} | device={device}"
    )
    if rank == 0:
        print(f"WORLD_SIZE={world_size}", flush=True)


def demo_barrier(rank, world_size):
    dist.barrier()
    start = time.perf_counter()

    if rank == 1:
        time.sleep(2)

    elapsed = time.perf_counter() - start
    print(f"Rank {rank} reached barrier at ~{elapsed:.2f}s", flush=True)

    dist.barrier()

    elapsed = time.perf_counter() - start
    print(f"Rank {rank} passed barrier at  ~{elapsed:.2f}s", flush=True)


def demo_broadcast(rank, world_size, device):
    tensor = (
        torch.tensor([10.0, 20.0], device=device)
        if rank == 0
        else torch.tensor([0.0, 0.0], device=device)
    )

    ordered_print(rank, world_size, f"[Rank {rank}] BEFORE: {fmt(tensor)}")
    dist.broadcast(tensor, src=0)
    ordered_print(rank, world_size, f"[Rank {rank}] AFTER : {fmt(tensor)}")


def demo_reduce(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    tensor = (
        torch.tensor([1.0, 2.0], device=device)
        if rank == 0
        else torch.tensor([3.0, 4.0], device=device)
    )

    ordered_print(rank, world_size, f"[Rank {rank}] BEFORE: {fmt(tensor)}")
    dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)

    if rank == 0:
        print(f"[Rank 0] REDUCE RESULT: {fmt(tensor)}", flush=True)
    dist.barrier()


def demo_all_reduce(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    tensor = (
        torch.tensor([1.0, 2.0], device=device)
        if rank == 0
        else torch.tensor([3.0, 4.0], device=device)
    )

    ordered_print(rank, world_size, f"[Rank {rank}] BEFORE: {fmt(tensor)}")
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    ordered_print(rank, world_size, f"[Rank {rank}] AFTER : {fmt(tensor)}")


def demo_all_gather(rank, world_size, device):
    tensor = torch.tensor([float((rank + 1) * 10)], device=device)
    outputs = [torch.zeros_like(tensor) for _ in range(world_size)]

    ordered_print(rank, world_size, f"[Rank {rank}] LOCAL INPUT: {fmt(tensor)}")
    dist.all_gather(outputs, tensor)

    gathered = [x.item() for x in outputs]
    ordered_print(rank, world_size, f"[Rank {rank}] GATHERED   : {gathered}")


def demo_reduce_scatter(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    input_tensor = (
        torch.tensor([1.0, 2.0, 3.0, 4.0], device=device)
        if rank == 0
        else torch.tensor([10.0, 20.0, 30.0, 40.0], device=device)
    )
    output = torch.empty(2, device=device)

    ordered_print(rank, world_size, f"[Rank {rank}] INPUT : {fmt(input_tensor)}")
    dist.reduce_scatter_tensor(output, input_tensor, op=dist.ReduceOp.SUM)
    ordered_print(rank, world_size, f"[Rank {rank}] OUTPUT: {fmt(output)}")


def demo_all_to_all(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    input_tensor = (
        torch.tensor([10.0, 11.0], device=device)
        if rank == 0
        else torch.tensor([20.0, 21.0], device=device)
    )
    output = torch.empty_like(input_tensor)

    ordered_print(rank, world_size, f"[Rank {rank}] INPUT : {fmt(input_tensor)}")
    dist.all_to_all_single(output, input_tensor)
    ordered_print(rank, world_size, f"[Rank {rank}] OUTPUT: {fmt(output)}")


def demo_manual_grad_sync(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    torch.manual_seed(1234)
    model = nn.Linear(2, 1, bias=False).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

    if rank == 0:
        x = torch.tensor([[1.0, 0.0], [0.0, 1.0]], device=device)
        y = torch.tensor([[1.0], [2.0]], device=device)
    else:
        x = torch.tensor([[2.0, 1.0], [1.0, 3.0]], device=device)
        y = torch.tensor([[3.0], [5.0]], device=device)

    optimizer.zero_grad()
    pred = model(x)
    loss = F.mse_loss(pred, y)
    loss.backward()

    grad = model.weight.grad

    ordered_print(
        rank,
        world_size,
        f"[Rank {rank}] local loss={loss.item():.6f} | LOCAL GRAD={fmt(grad)}"
    )

    dist.all_reduce(grad, op=dist.ReduceOp.SUM)
    grad /= world_size

    ordered_print(rank, world_size, f"[Rank {rank}] SYNCED GRAD={fmt(grad)}")

    optimizer.step()
    ordered_print(rank, world_size, f"[Rank {rank}] UPDATED WEIGHT={fmt(model.weight)}")


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--demo",
        required=True,
        choices=[
            "identity",
            "barrier",
            "broadcast",
            "reduce",
            "all_reduce",
            "all_gather",
            "reduce_scatter",
            "all_to_all",
            "manual_grad_sync",
        ],
    )
    return parser.parse_args()


def main():
    args = parse_args()
    rank, local_rank, world_size, device = setup()

    try:
        if args.demo == "identity":
            demo_identity(rank, local_rank, world_size, device)
        elif args.demo == "barrier":
            demo_barrier(rank, world_size)
        elif args.demo == "broadcast":
            demo_broadcast(rank, world_size, device)
        elif args.demo == "reduce":
            demo_reduce(rank, world_size, device)
        elif args.demo == "all_reduce":
            demo_all_reduce(rank, world_size, device)
        elif args.demo == "all_gather":
            demo_all_gather(rank, world_size, device)
        elif args.demo == "reduce_scatter":
            demo_reduce_scatter(rank, world_size, device)
        elif args.demo == "all_to_all":
            demo_all_to_all(rank, world_size, device)
        elif args.demo == "manual_grad_sync":
            demo_manual_grad_sync(rank, world_size, device)
    finally:
        cleanup()


if __name__ == "__main__":
    main()

## 4. Sanity Check — Who Am I?

Before moving any data, verify that every process has the expected identity and GPU assignment.


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo identity

Expected mental model:

```text
Rank 0 → GPU 0
Rank 1 → GPU 1
WORLD_SIZE = 2
```

> **Note:** Output ordering from multiple processes is not naturally deterministic. Most demos use `ordered_print()` so that the output is easier to teach from.


# 5. The Collective Contract

This is one of the most important ideas in the session.

A collective operation is not a function that each rank can call independently whenever it wants. Participating ranks must execute a compatible communication sequence.

Conceptually:

```text
Rank 0: all_reduce(A) ─────┐
                           ├── same collective
Rank 1: all_reduce(B) ─────┘
```

A mismatch such as:

```text
Rank 0: all_reduce(...)
Rank 1: broadcast(...)
```

or a situation where one rank enters a collective and another rank never reaches it can result in a **hang, timeout, or other failure depending on the mismatch**.

### Simple Rule

> **Collectives are group operations. Think about the whole group, not one rank in isolation.**

Tensor metadata must also be compatible with the selected collective: shape, dtype, device placement, split sizes, and other requirements depend on the specific operation.

### DO NOT RUN — Intentionally Broken Example

```python
if rank == 0:
    dist.all_reduce(tensor)
else:
    dist.broadcast(tensor, src=0)
```

The key lesson is that distributed debugging is different: each process may look reasonable in isolation while the **global communication schedule** is invalid.


# 6. `dist.barrier()` — Synchronization Point

`barrier()` does not move a tensor like `AllReduce`. It is a synchronization primitive.

Conceptually:

```text
Rank 0 ────────┐
               │
               ├── BARRIER ───→ continue
               │
Rank 1 ────────────────┘
```

No participating rank proceeds past the barrier until all participating ranks reach it.

In the next demo, Rank 1 intentionally sleeps for about two seconds. Rank 0 arrives early but cannot pass the barrier until Rank 1 arrives.


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo barrier

### What Should You Observe?

Approximately:

```text
Rank 0 reached barrier at ~0.00s
Rank 1 reached barrier at ~2.00s

Rank 0 passed barrier at ~2.00s
Rank 1 passed barrier at ~2.00s
```

The exact timing and print order may differ slightly, but the idea is the same:

> **A fast rank waits for the slow rank at the barrier.**

### Note

`barrier()` is useful for experiments, coordination, and debugging, but excessive barriers can remove useful overlap and introduce unnecessary synchronization overhead.


# 7. Broadcast — One Rank → Everyone

## Before Running: Predict the Output

```text
Rank 0 → [10, 20]
Rank 1 → [ 0,  0]
```

We call:

```python
dist.broadcast(tensor, src=0)
```

What should each rank contain afterward?

### Mental Model

```text
          Rank 0
        [10, 20]
        /      \
       v        v
    Rank 0    Rank 1
  [10,20]    [10,20]
```

> **Important:** `src=0` means **Rank 0**, not a physical GPU identifier.


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo broadcast

### Outcome

```text
BEFORE
Rank 0 → [10,20]
Rank 1 → [0,0]

AFTER BROADCAST(src=0)
Rank 0 → [10,20]
Rank 1 → [10,20]
```

**Pattern:** One-to-all.

Broadcast can be useful when one rank owns initial state, metadata, or a tensor that must be replicated to the rest of the group.


# 8. Reduce — Everyone Contributes → One Rank Gets the Reduced Result

Inputs:

```text
Rank 0 → [1,2]
Rank 1 → [3,4]
```

We use SUM:

```python
dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)
```

Element-wise:

```text
[1,2]
+
[3,4]
=
[4,6]
```

The final reduced result is meaningful on the **destination rank**.

> `dst=0` means Rank 0.


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo reduce

### Mental Model

```text
Rank 0 [1,2] ───┐
                 ├── SUM ───→ Rank 0 [4,6]
Rank 1 [3,4] ───┘
```

**Pattern:** Many-to-one + reduction.

### Teaching Note

Do not treat the contents of non-destination ranks after `reduce()` as the final result. The semantic result we care about is on `dst`.


# 9. AllReduce — Everyone Contributes → Everyone Gets the Reduced Result

Use the same inputs:

```text
Rank 0 → [1,2]
Rank 1 → [3,4]
```

Instead of sending the final result only to Rank 0, every participating rank receives the reduced result:

```python
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
```


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo all_reduce

### Result

```text
Rank 0 → [4,6]
Rank 1 → [4,6]
```

### Reduce vs AllReduce

| Operation | Who contributes? | Who gets the final reduced result? |
|---|---|---|
| Reduce | All participating ranks | One destination rank |
| AllReduce | All participating ranks | Every participating rank |

This pattern is extremely important in distributed training because every worker can compute local gradients and then receive synchronized gradients before applying the same model update.


# 10. AllGather — Everyone Contributes a Piece → Everyone Gets All Pieces

There is no SUM here.

Inputs:

```text
Rank 0 → [10]
Rank 1 → [20]
```

After `AllGather`:

```text
Rank 0 → [10,20]
Rank 1 → [10,20]
```

Each rank contributes one piece, and every rank receives the collection of all pieces.


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo all_gather

### AllGather vs AllReduce

This distinction is easy to confuse:

```text
AllGather
---------
R0: [10]
R1: [20]

→ both get [10,20]
```

No element-wise reduction occurs.

By contrast:

```text
AllReduce SUM
-------------
R0: [10]
R1: [20]

→ both get [30]
```

### Mental Model

- **AllGather:** collect pieces.
- **AllReduce:** combine values with a reduction operation and give the result to everyone.


# 11. ReduceScatter — Reduce First, Then Distribute Pieces of the Result

This operation is especially important because it connects directly to Ring AllReduce.

Inputs:

```text
Rank 0 → [ 1,  2,  3,  4]
Rank 1 → [10, 20, 30, 40]
```

An element-wise SUM gives:

```text
[11, 22, 33, 44]
```

But instead of giving the entire reduced tensor to every rank:

```text
Rank 0 → [11,22]
Rank 1 → [33,44]
```

Conceptually:

```text
REDUCE
   +
SCATTER
```


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo reduce_scatter

### Key Idea

After `ReduceScatter`:

- The reduction has happened across the ranks.
- Each rank keeps **one part** of the reduced result.

This is different from AllReduce, where every rank receives the complete reduced result.


# 12. A Very Important Identity

Semantically:

```text
AllReduce
   =
ReduceScatter
   +
AllGather
```

Why?

### Step 1 — ReduceScatter

All ranks contribute to the reduction, then different ranks own different reduced chunks.

```text
R0 owns reduced C0
R1 owns reduced C1
R2 owns reduced C2
R3 owns reduced C3
```

### Step 2 — AllGather

Each rank shares its reduced chunk with the others.

Eventually every rank owns:

```text
[C0, C1, C2, C3]
```

That is the AllReduce result.

> **Precision note:** This semantic decomposition is extremely useful for understanding Ring AllReduce, but it does **not** mean NCCL must always implement every AllReduce using the same algorithm. NCCL may choose different algorithms depending on topology, message size, hardware, and other runtime factors.


# 13. AllToAll — Every Rank Sends Different Pieces to Different Ranks

Inputs:

```text
Rank 0 → [10,11]
Rank 1 → [20,21]
```

Interpret them as:

```text
Rank 0:
10 → destination Rank 0
11 → destination Rank 1

Rank 1:
20 → destination Rank 0
21 → destination Rank 1
```

After AllToAll:

```text
Rank 0 → [10,20]
Rank 1 → [11,21]
```


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo all_to_all

### Why Should We Care?

`AllToAll` is especially important in patterns such as:

- Expert Parallelism
- Mixture-of-Experts (MoE)
- Some tensor and data redistribution patterns

We will not deep dive into MoE here. The main idea is:

> **AllToAll routes different pieces to different destinations.**


# 14. Collective Cheat Sheet

| Collective | Simple mental model | Final ownership |
|---|---|---|
| Broadcast | One → All | Everyone gets the root rank's data |
| Reduce | All → One + reduction | One destination gets the reduced result |
| AllReduce | All → All + reduction | Everyone gets the reduced result |
| AllGather | Each rank contributes a piece | Everyone gets all pieces |
| ReduceScatter | Reduce + split result | Each rank gets one reduced piece |
| AllToAll | Everyone sends different pieces everywhere | Each rank receives its destined pieces |
| Barrier | Synchronize only | No tensor result |

### Quick Test

Given:

```text
Rank 0 → [1]
Rank 1 → [2]
```

What happens with:

- AllReduce SUM?
- AllGather?
- Reduce SUM to Rank 0?

Answer:

```text
AllReduce SUM:
R0 [3]
R1 [3]

AllGather:
R0 [1,2]
R1 [1,2]

Reduce SUM to R0:
R0 [3]
```


# 15. Ring AllReduce — Now We Know WHAT, Let's Discuss HOW

At this point, the semantics of `dist.all_reduce()` are clear:

> Every rank contributes, and every rank receives the reduced result.

Now the question is: how can this be implemented efficiently?

One well-known plan is **Ring AllReduce**.

We will use 4 ranks conceptually:

```text
R0 → R1 → R2 → R3 → R0
```

Each rank owns a tensor split into chunks:

| Rank | C0 | C1 | C2 | C3 |
|---|---:|---:|---:|---:|
| R0 | 1 | 10 | 100 | 1000 |
| R1 | 2 | 20 | 200 | 2000 |
| R2 | 3 | 30 | 300 | 3000 |
| R3 | 4 | 40 | 400 | 4000 |

The target AllReduce SUM is:

```text
[10, 100, 1000, 10000]
```

on **every rank**.


## 15.1 Understand One Chunk First — C0

Do not look at all four chunks at once.

Start with C0 only:

```text
R0 has 1
R1 has 2
R2 has 3
R3 has 4
```

The partial sum moves around the ring:

```text
R0 sends 1 → R1
R1: 1 + 2 = 3

R1 sends 3 → R2
R2: 3 + 3 = 6

R2 sends 6 → R3
R3: 6 + 4 = 10
```

So the reduced value for C0 is:

```text
C0 = 10
```

The important idea is that a partial result moves and is combined with each rank's local contribution.


## 15.2 Now Let Different Chunks Move at the Same Time

If the whole tensor moved as one sequential object, many communication resources could remain underutilized.

Instead, the tensor is split into chunks so that different pieces can make progress concurrently.

### ReduceScatter — Round 1

```text
R0 sends C0=1    → R1 → 1+2       = 3
R1 sends C1=20   → R2 → 20+30     = 50
R2 sends C2=300  → R3 → 300+400   = 700
R3 sends C3=4000 → R0 → 4000+1000 = 5000
```

### Round 2

```text
C0: 3    → R2 → +3    = 6
C1: 50   → R3 → +40   = 90
C2: 700  → R0 → +100  = 800
C3: 5000 → R1 → +2000 = 7000
```

### Round 3

```text
C0: 6    → R3 → +4    = 10
C1: 90   → R0 → +10   = 100
C2: 800  → R1 → +200  = 1000
C3: 7000 → R2 → +3000 = 10000
```

The reduced chunks are now distributed:

```text
R3 owns C0 = 10
R0 owns C1 = 100
R1 owns C2 = 1000
R2 owns C3 = 10000
```

This is the **ReduceScatter phase**.


## 15.3 AllGather Phase

No additional reduction is required now.

Each rank owns one completed reduced chunk, and the goal is to distribute those chunks to every rank.

```text
R0 has C1
R1 has C2
R2 has C3
R3 has C0
```

Through ring exchanges, the completed chunks circulate until every rank owns:

```text
[C0, C1, C2, C3]
=
[10, 100, 1000, 10000]
```

Therefore:

```text
Ring AllReduce
=
Ring ReduceScatter
+
Ring AllGather
```

### Why Chunks Matter

Chunks are not required by the mathematics of reduction.

They help expose concurrency, pipeline communication, and use multiple communication resources more effectively.

> **Important:** Ring is an algorithm / communication plan, not a transport.  
> The physical path may involve PCIe, NVLink, or a network, while the NCCL transport mechanism may involve P2P, SHM, or NET depending on the environment.


# 16. The Training Connection — Why Does AllReduce Matter?

Now connect the communication primitive to training.

In data-parallel-style training:

```text
Same Model
   │
   ├── Rank 0 gets Batch A
   │       ↓
   │    Forward
   │       ↓
   │    Backward
   │       ↓
   │    Local Gradients
   │
   └── Rank 1 gets Batch B
           ↓
        Forward
           ↓
        Backward
           ↓
        Local Gradients
```

Because the local batches are different, local gradients will generally be different.

If each rank immediately executes `optimizer.step()`:

```text
Model on Rank 0 changes one way
Model on Rank 1 changes another way
```

The replicas can diverge.

Instead:

```text
Local Gradients
      ↓
AllReduce SUM
      ↓
Divide by WORLD_SIZE
      ↓
Same Averaged Gradient
      ↓
optimizer.step()
      ↓
Same Parameter Update
```


## 16.1 Manual Gradient Synchronization

We will use a tiny linear model.

Important conditions:

- Both ranks start with the same weights.
- Each rank receives a different local batch.
- Run `backward()`.
- Print gradients **before synchronization**.
- Run `all_reduce`.
- Divide by `world_size`.
- Print gradients **after synchronization**.
- Run the optimizer step and verify that updated weights are identical.

### Predict Before Running

Before AllReduce:

```text
Rank 0 gradient != Rank 1 gradient
```

After AllReduce + average:

```text
Rank 0 gradient == Rank 1 gradient
```


In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo manual_grad_sync

### What Just Happened?

We manually implemented a core part of distributed data-parallel training:

```python
loss.backward()

dist.all_reduce(param.grad, op=dist.ReduceOp.SUM)
param.grad /= world_size

optimizer.step()
```

### Important Precision Note

Dividing by `world_size` gives the correct average of local mean gradients in this demo because every rank uses the **same local batch size**.

If local batch sizes differ, the correct global-example average requires appropriate weighting. A simple divide-by-world-size is not always the correct sample-weighted global mean.


# 17. Bridge to DDP

When you later see:

```python
from torch.nn.parallel import DistributedDataParallel as DDP

model = DDP(
    model,
    device_ids=[local_rank]
)
```

it should no longer look like magic.

Conceptually:

```text
Forward
   ↓
Backward
   ↓
Gradient becomes ready
   ↓
DDP coordinates gradient synchronization
   ↓
Collective communication through ProcessGroupNCCL
   ↓
NCCL moves/reduces the data
   ↓
Synchronized gradients
```

A real DDP implementation is more sophisticated than calling one Python `all_reduce()` per parameter.

It uses mechanisms such as **gradient buckets** so communication can begin while backward computation is still in progress.

That creates a natural bridge to a future session on DDP internals.


# 18. Common Mistakes & Debugging Notes

### 1. Different collective order across ranks

```text
Rank 0:
all_reduce()
broadcast()

Rank 1:
broadcast()
all_reduce()
```

This is a problem.

The collective schedule must remain compatible across participating ranks.

---

### 2. One rank never reaches the collective

A rank may have:

- crashed
- run out of memory
- become stuck in a dataloader or another code path
- entered a different branch

Other ranks can remain blocked waiting for it.

---

### 3. Confusing rank with GPU index

```python
src=0
dst=0
```

These are ranks, not physical GPU IDs.

---

### 4. Shape / split mismatches

Different collectives have different input/output contracts.

`ReduceScatter` and `AllToAll`, in particular, require careful thinking about how the data is partitioned.

---

### 5. Too many barriers

`barrier()` is useful for teaching and debugging, but production code should not add unnecessary synchronization because it can reduce parallelism.

---

### 6. Printing from every rank

Multi-process logs can interleave.

For teaching and debugging, use:

- rank-prefixed logs
- barriers when deterministic presentation is needed
- Rank 0-only logging when appropriate


# 19. Topics We Intentionally Did NOT Deep Dive Into

## Gather / Scatter

Useful operations, but the core communication mental model is already covered by the collectives above.

## `async_op=True`

Some PyTorch distributed APIs support asynchronous work handles. This is important for communication/computation overlap, but it deserves a dedicated discussion of synchronization and CUDA stream semantics.

## Custom Process Groups

Not every communication operation must include the entire world. Subgroups become especially important in Tensor Parallelism, Pipeline Parallelism, and Expert Parallelism.

## Performance Benchmarking

Later we can measure:

- latency
- effective bandwidth
- message-size effects
- `nccl-tests`
- communication/computation overlap

The first priority here is understanding the semantics.

## Advanced Algorithms

Ring is not the only possible communication algorithm. NCCL can use different execution plans depending on the environment. Ring is used here because it provides a strong mental model for chunked collective communication.


# 20. Final Mental Model

```text
Application needs a communication result
              ↓
Choose a Collective
              ↓
WHAT result?
Broadcast / AllReduce / AllGather / ...
              ↓
NCCL chooses an execution strategy
              ↓
HOW to move it?
Algorithm + chunks + channels + protocol + transport
              ↓
Physical interconnect
              ↓
GPU kernels / network progress
              ↓
Bytes move
```

### The Key Distinction

**Collective = semantics.**

It answers:

> What result should the group produce?

Ring, Tree, transports, protocols, and channels are execution and implementation choices below the collective abstraction.


# 21. Session Outcomes Checklist

After finishing the notebook, try to answer these questions without looking back:

- [ ] What is the difference between Reduce and AllReduce?
- [ ] What is the difference between AllGather and AllReduce?
- [ ] Why is ReduceScatter useful?
- [ ] What can happen if ranks call collectives in an incompatible order?
- [ ] What does `barrier()` guarantee?
- [ ] Why can local gradients differ between ranks?
- [ ] Why does gradient synchronization keep model replicas aligned?
- [ ] Why is `AllReduce = ReduceScatter + AllGather` a useful mental model?
- [ ] Why is Ring an algorithm rather than a transport?
- [ ] Why does `backend="nccl"` not tell you which physical interconnect is being used?

If these answers are clear, you are ready to study **DDP internals** without treating `DistributedDataParallel` as a black box.


# References

For follow-up reading:

- **PyTorch Distributed documentation** — `torch.distributed` collectives and process groups  
  https://docs.pytorch.org/docs/stable/distributed.html

- **PyTorch DistributedDataParallel documentation** — gradient synchronization and DDP behavior  
  https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html

- **NVIDIA NCCL documentation** — collective communication primitives and NCCL concepts  
  https://docs.nvidia.com/deeplearning/nccl/

> This notebook is intentionally teaching-oriented: examples are small and explicit so the communication semantics remain visible before moving into performance and framework internals.
